# 43 - Humanoid Manufacturing And Cost

## What / Why / How

**What we are trying to do:** Reason about manufacturability, reliability, cost, serviceability, and fleet learning.

**Why this matters:** Optimus-like robots are not just lab prototypes; they require supply chains, calibration, testing, and maintenance.

**How we will do it:** Build a simple bill-of-materials model, failure-rate estimate, and fleet data flywheel simulation.

## Manufacturing And Fleet Learning

Tesla's advantage, if Optimus succeeds, would likely be manufacturing plus AI data loops. This notebook models those pressures at toy scale.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
bom = {
    "actuators": 40 * 180,
    "sensors": 900,
    "compute": 1200,
    "battery_power": 800,
    "structure": 1500,
    "hands": 2200,
    "assembly_test": 2500,
}
total = sum(bom.values())
for k, v in bom.items():
    print(f"{k:14}: ${v:,.0f}")
print("toy BOM total:", f"${total:,.0f}")

In [ ]:
units = np.array([10, 100, 1000, 10000], dtype=float)
learning_rate = 0.85  # cost falls to 85% each doubling, toy assumption
cost = total * learning_rate ** (np.log2(units / units[0]))
for u, c in zip(units, cost):
    print(f"units={int(u):5} estimated unit cost=${c:,.0f}")

In [ ]:
fleet_size = 100
tasks_per_robot_per_day = 80
failure_rate = 0.08
days = 30
failures = []
for day in range(days):
    attempts = fleet_size * tasks_per_robot_per_day
    failures.append(attempts * failure_rate)
    failure_rate *= 0.97  # learning from failures
print("first day failures:", round(failures[0]))
print("day 30 failures:", round(failures[-1]))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(np.arange(days) + 1, failures)
    plt.xlabel("day")
    plt.ylabel("failures/day")
    plt.grid(True, alpha=0.3)
    plt.title("Toy fleet learning curve")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add service labor cost.
2. Add actuator replacement rate.
3. Explain why fleet data can be more valuable than a single impressive demo.